# 02 · Seasonal-naive baseline and the metric
Frame first (Section 07): **weekly SKU demand, 8-week horizon, WAPE as the primary metric, bias as the secondary check.** The baseline is the bar every model must clear.

In [1]:
import sys, json, warnings; warnings.filterwarnings('ignore')
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 20); pd.set_option('display.max_colwidth', 80)

In [2]:
from src import pipeline, config as C
from src.features import Panel
from src.forecast import seasonal_naive, wape, bias, mape, backtest_origins
P = Panel(pipeline.run(save=False, verbose=False)['weekly'])
as_of = pd.Timestamp(json.loads((ROOT/'outputs/metrics.json').read_text())['as_of'])
last_known = int(np.searchsorted(P.weeks, as_of) - 1)
print('Horizon:', C.HORIZON_WEEKS, 'weeks | last training week:', P.weeks[last_known].date(), '| origins:', [str(P.week_date(o).date()) for o in backtest_origins(last_known)])

Horizon: 8 weeks | last training week: 2025-11-24 | origins: ['2025-01-20', '2025-02-17', '2025-03-17', '2025-04-14', '2025-05-12', '2025-06-09', '2025-07-07', '2025-08-04', '2025-09-01', '2025-09-29']


## Metric definitions (Appendix B)
- **WAPE** = Σ|actual − forecast| / Σ actual. Robust to low-volume SKUs.
- **Bias** = Σ(forecast − actual) / Σ actual. Positive = over-forecasting.
- **MAPE** reported as a secondary figure only.

## Baseline: same week last year, evaluated on the same rolling origins the model uses

In [3]:
rows = []
for o in backtest_origins(last_known):
    sn = seasonal_naive(P, o)
    for h in range(1, C.HORIZON_WEEKS+1):
        rows.append(pd.DataFrame({'fold': o, 'h': h, 'actual': P.Y[:, o+h], 'baseline': sn[h]}))
b = pd.concat(rows)
print('Seasonal-naive  WAPE %.3f | bias %+.3f | MAPE %.3f' % (wape(b.actual, b.baseline), bias(b.actual, b.baseline), mape(b.actual, b.baseline)))
b.groupby('fold').apply(lambda g: wape(g.actual, g.baseline)).rename('baseline_wape').round(3).to_frame().T

Seasonal-naive  WAPE 0.105 | bias +0.003 | MAPE 0.133


fold,55,59,63,67,71,75,79,83,87,91
baseline_wape,0.096,0.096,0.1,0.096,0.1,0.11,0.117,0.121,0.114,0.11


The baseline already misses by only ~10%: demand repeats year to year. Any model has to beat that on the same test windows to be worth using.